In [2]:
import os
import time
import torch
import numpy as np
import torch.nn.functional as F
from utils import splits_regression
from torch_geometric.loader import DataLoader
from torch_geometric.datasets import WikipediaNetwork
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [3]:
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class Net2(torch.nn.Module):
    def __init__(self, num_features, hidden, num_layers):
        super(Net2, self).__init__()
        self.num_layers = num_layers
        self.conv = torch.nn.ModuleList()
        self.conv.append(GCNConv(num_features, hidden))
        for i in range(self.num_layers - 1):
            self.conv.append(GCNConv(hidden, hidden))
        self.lt1 = torch.nn.Linear(hidden, 1)

    def reset_parameters(self):
        for module in self.conv:
            module.reset_parameters()
        self.lt1.reset_parameters()

    def forward(self, x, edge_index):
        for i in range(self.num_layers):
            x = self.conv[i](x, edge_index)
            x = F.elu(x)
            x = F.dropout(x, training=self.training)
        x = self.lt1(x)
        return x

In [4]:
dataset_names = ['chameleon', "squirrel", "crocodile"]
for dataset_name in dataset_names:
    dataset = WikipediaNetwork(root='./dataset', name=dataset_name, geom_gcn_preprocess=False)
    data = splits_regression(dataset[0], 0.2, 0.3)
    model = Net2(data.x.shape[1], 512, 2).to(device)
    loss_fn = torch.nn.L1Loss()
    model.reset_parameters()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
    all_loss = []
    avg_time = 0
    for run in range(20):
        best_val_loss = 100000
        model.reset_parameters()
        data = data.to(device)
        for epoch in range(100):
            model.train()
            optimizer.zero_grad()
            out = model(data.x, data.edge_index)
            loss = loss_fn(out[data.train_mask].view(-1, 1), data.y[data.train_mask].view(-1, 1))
            loss.backward()
            optimizer.step()

            model.eval()
            with torch.no_grad():
                out = model(data.x, data.edge_index)
                val_loss = loss_fn(out[data.val_mask].view(-1, 1), data.y[data.val_mask].view(-1, 1))
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    #save model
                    torch.save(model.state_dict(), f'save/node_reg/baselines/baseline_{dataset_name}.pt')
        start = time.time()
        out = model(data.x, data.edge_index)
        avg_time += time.time() - start
        test_loss = loss_fn(out[data.test_mask].view(-1, 1), data.y[data.test_mask].view(-1, 1))
        all_loss.append(test_loss.item()/data.y[data.test_mask].std().item())

    print(f"##########{dataset_name}##########")
    print(all_loss)
    print(sum(all_loss)/len(all_loss))
    print(avg_time/20)
    top_loss = sorted(all_loss)[:10]

    if not os.path.exists(f"results/node_reg_baselines.csv"):
        with open(f"results/node_reg_baselines.csv", 'w') as f:
            f.write('dataset,hidden,runs,num_layers,lr,avg_time,top_10_loss,best_loss\n')

    with open(f"results/node_reg_baselines.csv", 'a') as f:
        f.write(f"{dataset_name},512,20,2,0.01,{avg_time/20},{np.mean(top_loss)} +/- {np.std(top_loss)},{top_loss[0]}\n")

##########chameleon##########
[0.5590655885984109, 0.548963245917498, 0.5500804015203197, 0.54638418835221, 0.563922071716225, 0.541924323542634, 0.535912959768476, 0.5645106274619527, 0.5529355480933803, 0.5402507234003567, 0.5330691531668611, 0.5502630199713738, 0.5558243777499431, 0.5334693081988667, 0.5527804374938529, 0.542782007125544, 0.570642071428796, 0.5978041661034107, 0.5367252834656031, 0.5550089664368283]
0.551615923475627
0.012350201606750488
##########squirrel##########
[0.6399740050679493, 0.6467953591064195, 0.6572444118864564, 0.6442376152551185, 0.6440190611339942, 0.6549523519324775, 0.6510510689264805, 0.6403780134962789, 0.6403263886836231, 0.6742421198265025, 0.644993121913153, 0.6558507462645906, 0.6688625827852908, 0.6563809562468135, 0.636423675438753, 0.6625641513214522, 0.6424027549942238, 0.6678188489410307, 0.6558382827544113, 0.6550684737076463]
0.6519711994841332
0.007388401031494141
##########crocodile##########
[0.37955094162031644, 0.3944405955808431